In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
%sql
SELECT * FROM dbw_gitarchive.silver.github_events LIMIT 2
dirty_df = spark.sql("select * FROM dbw_gitarchive.silver.github_events")
dirty_df.show(10)
#Whether there are duplicates or not, we will still clean it up for the future refrence
#First, remove the records that have nulls in id, as that is not useful for us

no_null_df = dirty_df.filter(F.col("event_id").isNotNull())

now, we check for the duplicates

def remove_duplicates(df, primary_col, order_col):
    window_dup = Window.partitionBy(primary_col).orderBy(order_col)
    df = df.withColumn("rank", F.rank().over(window_dup))
    df = df.filter(F.col("rank") == 1)
    df = df.drop("rank")
    return df
silver_cleaned_df = remove_duplicates(no_null_df, "event_id", "processed_at")
silver_cleaned_df.show()
### Now we check for the column TYPE
silver_cleaned_df = silver_cleaned_df.withColumn("event_type",\
    F.trim(F.col("event_type")))\
    .filter((F.col("event_type").isNotNull()) & (F.col("event_type") != ""))
## Now, the interestinf part is ACTOR column
from pyspark.sql.types import StructType, StructField, StringType, LongType
#Since the actor column is not a normal string, I will have to convert it to a struct

#Defining internal struct schema for actor JSON string

actor_schema = StructType([\
    StructField("id", LongType(), True),\
    StructField("login", StringType(), True),\
    StructField("display_login", StringType(), True),\
    StructField("url", StringType(), True),\
    StructField("avatar_url", StringType(), True)\
    ])
df_parsed = silver_cleaned_df.withColumn("actor_struct", F.from_json(F.col("actor"), actor_schema))
df_parsed.select("actor_struct.*").show(5, truncate=False)
#Since, that worked, Implementing it in the silver df
silver_cleaned_df = silver_cleaned_df.withColumn("actor_struct", F.from_json(F.col("actor"), actor_schema))

# I will only be using the id and login (username) as rest are not very needed for next level

silver_cleaned_df = silver_cleaned_df.withColumn("actor_id", F.col("actor_struct.id"))\
    .withColumn("actor_username", F.col("actor_struct.login"))
silver_cleaned_df.show()
### Now cleaning the actor_id and actor_username

silver_cleaned_df = silver_cleaned_df.filter(\
    (F.col("actor_id").isNotNull()) 
    & (F.col("actor_id") > 0)                 # Consistency: IDs must be valid positive numbers
    & (F.col("actor_username").isNotNull())
    & (F.col("actor_username") != ""))
#And to be future proof
silver_cleaned_df = silver_cleaned_df.withColumn("actor_username", F.trim(F.col("actor_username")))
Now for other column
silver_cleaned_df.display(5)
### Now time for Repo Column

def extract_nested_json_fields(df, json_col, field1, col1_name, field2, col2_name):
   
    df = (df
          .withColumn(col1_name, F.col(f"{json_col}.{field1}"))
          .withColumn(col2_name, F.col(f"{json_col}.{field2}"))
         )
    return df
def trim_clean_json(df, col1, col2):
    df = df.filter(\
        (F.col(f"{col1}").isNotNull()) 
        & (F.col(f"{col1}") > 0)                 # Consistency: IDs must be valid positive numbers
        & (F.col(f"{col2}").isNotNull())
        & (F.col(f"{col2}") != ""))
    return df
repo_schema = StructType([
    StructField("id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("url", StringType(), True)
])

silver_cleaned_df = silver_cleaned_df.withColumn("repo_struct", F.from_json(F.col("repo"), repo_schema))

silver_cleaned_df = extract_nested_json_fields(
    df=silver_cleaned_df, 
    json_col="repo_struct", 
    field1="id",   col1_name="repo_id", 
    field2="name", col2_name="repo_name"
)

#Now filtering the id and name

silver_cleaned_df = trim_clean_json(
    df = silver_cleaned_df,
    col1 = "repo_id",
    col2 = "repo_name"
)
silver_cleaned_df.display(10)
### The column Payload is the most complex column in this dataset. The real question is whether or not to extract it
#I will proceed without extracting it, but if extraction is needed ion future, here is the code
# ============================================================================
# REFERENCE CODE ONLY: SAMPLE PAYLOAD EXTRACTION
# Use this pattern when creating specialized downstream tables (e.g., Pull Requests analysis)
# ============================================================================
"""
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, StructType
import pyspark.sql.functions as F

# 1. Define a targeted schema for the specific field
# Note: PySpark safely ignores any JSON keys that are not explicitly defined here.
payload_sample_schema = StructType([
    # Fields typical for PushEvents
    StructField("size", IntegerType(), True),          # Number of commits in the push
    StructField("distinct_size", IntegerType(), True), # Number of unique commits
    
    # Fields typical for PullRequestEvents (nested object)
    StructField("action", StringType(), True),         # e.g., "opened", "closed"
    StructField("pull_request", StructType([
        StructField("id", LongType(), True),
        StructField("number", IntegerType(), True),
        StructField("state", StringType(), True),      # e.g., "open", "closed"
        StructField("title", StringType(), True)
    ]), True)
])

# 2. Example Extraction Flow:
# Parsed column turns into a structured object, allowing clean dot-notation access.
df_extracted = (
    silver_cleaned_df
    .withColumn("payload_struct", F.from_json(F.col("payload"), payload_sample_schema))
    .withColumn("push_commit_count", F.col("payload_struct.size"))
    .withColumn("pr_action", F.col("payload_struct.action"))
    .withColumn("pr_title", F.col("payload_struct.pull_request.title"))
    .drop("payload_struct")
)
"""
# ============================================================================
#Just basic cleaning of the payload column
silver_cleaned_df = silver_cleaned_df.withColumn("payload", F.trim(F.col("payload")))

silver_cleaned_df = silver_cleaned_df.filter(
    (F.col("payload").isNotNull()) &
    (F.col("payload") != "")
)

silver_cleaned_df.display(5)
### Now time for the clening of is_public

silver_cleaned_df.select("is_public").distinct()

#This column looks clean but lets prepare for the future


silver_cleaned_df = (
    silver_cleaned_df
    .withColumn(
        "is_public",
        F.when(
            F.upper(F.trim(F.col("is_public"))).isin("T", "TRUE", "1"), 
            True
        )
        .when(
            F.upper(F.trim(F.col("is_public"))).isin("F", "FALSE", "0"), 
            False
        )
        .otherwise(None) # Catches corrupt strings or nulls safely
    )
    .filter(F.col("is_public").isNotNull()) # Data quality check
)

silver_cleaned_df.select("is_public").distinct().display()

#Now, we clean the org column, since there are a lot of nulls, we cannot directly use filter as it will delete most of the individual columns

org_schema = StructType([
    StructField("id", LongType(), True),
    StructField("log_in", StringType(), True)
])

# 2. Parse the JSON string into a struct column
silver_cleaned_df = silver_cleaned_df.withColumn("org_struct", F.from_json(F.col("org"), org_schema))

# 3. Safely Extract, Trim, and Standardize inline
silver_cleaned_df = (
    silver_cleaned_df
    .withColumn("org_id", F.col("org_struct.id"))
    # Trim the name, and if it's an empty string, turn it into a proper null
    .withColumn(
        "org_name", 
        F.when(F.trim(F.col("org_struct.log_in")) != "", F.trim(F.col("org_struct.log_in")))
         .otherwise(None)
    ))

silver_cleaned_df = (
    silver_cleaned_df
    .withColumn("org_id", F.when(F.col("org_id") > 0, F.col("org_id")).otherwise(None))
    .withColumn("org_name", F.when(F.col("org_id").isNotNull(), F.col("org_name")).otherwise(None))
)

# 5. Drop intermediary structural columns
silver_cleaned_df = silver_cleaned_df.drop("org", "org_struct")
silver_cleaned_df.display(5)
Now for the dates

silver_cleaned_df.display(5)
# Wrap in outer parentheses for implicit line continuation


silver_cleaned_df = (
    silver_cleaned_df
    # 1. Convert the raw string timestamp into a highly optimized TimestampType
    .withColumn("created_at", F.to_timestamp(F.col("event_created_at")))
    
    # 2. Overwrite the string 'event_date' with a proper, strongly-typed DateType
    .withColumn("event_date", F.to_date(F.col("created_at")))
    
    # 3. Standardize and rename the audit infrastructure trail
    .withColumnRenamed("source_file", "audit_source_file")
    .withColumn("audit_processed_at", F.current_timestamp())
    
    # 4. Drop the old unoptimized duplicate text columns
    .drop("event_created_at", "processed_at")
)
silver_cleaned_df.display(5)
NOw, lets drop the columns that we dont need

# The final cleanup: Keeping the flat columns, dropping the structural clutter
silver_cleaned_df = silver_cleaned_df.drop(
    "actor", 
    "repo", 
    "actor_struct", 
    "repo_struct"
)
silver_cleaned_df.display(5)
#Finally adding to the delta table
(
    silver_cleaned_df.write
    .format("delta")
    .mode("append")
    .partitionBy("event_date")
    .save("dbfs:/mnt/outputs/silver_github_events")
)